In [1]:
import os
import pandas as pd
from pathlib import Path

DATA_FILENAME = "hotel_bookings.csv"

def find_data_file(filename: str = DATA_FILENAME) -> Path:
    """Tìm file CSV trong workspace (ưu tiên thư mục data/ cạnh notebook)."""
    notebook_dir = Path(os.environ.get("VSCODE_NOTEBOOK_DIR", Path.cwd()))

    search_roots = [
        notebook_dir / "data",
        notebook_dir,
        notebook_dir.parent / "data",
        notebook_dir.parent,
        notebook_dir.parent.parent,
        Path.cwd() / "data",
        Path.cwd(),
    ]

    #? Đi lên tối đa 6 cấp từ thư mục làm việc hiện tại
    path = Path.cwd()
    for _ in range(6):
        search_roots.extend([path / "data", path])
        path = path.parent

    checked = []
    seen = set()
    for root in search_roots:
        try:
            root = root.resolve()
        except OSError:
            continue
        if root in seen:
            continue
        seen.add(root)
        candidate = root / filename
        checked.append(str(candidate))
        if candidate.is_file():
            return candidate

    raise FileNotFoundError(
        f"Không tìm thấy '{filename}'.\n"
        f"Thư mục làm việc: {Path.cwd()}\n"
        f"Đã kiểm tra:\n  - " + "\n  - ".join(checked[-8:]) + "\n\n"
        "→ Đặt file tại Python/data/hotel_bookings.csv\n"
        "→ Chọn Local Python kernel (góc phải notebook), không dùng kernel từ xa."
    )

csv_path = find_data_file()
print(f"Đang đọc: {csv_path}")

df = pd.read_csv(csv_path)

#========================================================================================================

#* Xem thông tin cơ bản
print("=== data size ===")
print(f"Row: {df.shape[0]}, Col: {df.shape[1]}")

print("\n=== 10 first row ===")
print(df.head(10))

print("\n=== column info ===")
print(df.info())

print("\n=== data describe ===")
print(df.describe())


Đang đọc: C:\Users\ADMIN\OneDrive\Tài liệu\oneDrive\Desktop\DA\Project\Hotel Booking Demand\Python\data\hotel_bookings.csv
=== data size ===
Row: 119390, Col: 32

=== 10 first row ===
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   
5  Resort Hotel            0         14               2015               July   
6  Resort Hotel            0          0               2015               July   
7  Resort Hotel            0          9               2015               July   
8  Resort Hotel            1         85               2015               July   
9  R

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [2]:
list_null_columns = df.columns[df.isnull().any()].tolist()

print(list_null_columns)

import numpy as np

#* Bước 1: Biến tất cả chuỗi rỗng hoặc chỉ chứa khoảng trắng thành NaN (trên một bản sao)
df_checked = df.replace(r"^\s*$", np.nan, regex=True)

#* Bước 2: Thống kê lại
deep_null = df_checked.isnull().sum()
deep_null[deep_null > 0]

['children', 'country', 'agent', 'company']


children         4
country        488
agent        16340
company     112593
dtype: int64

In [3]:
# 1. Xóa cột 'company' do thiếu dữ liệu nghiêm trọng
df = df.drop(columns=["company"])

# 2. Xử lý các cột còn lại
df["children"] = df["children"].fillna(0).astype("int64")
df["country"] = df["country"].fillna("Unknown")
df["agent"] = df["agent"].fillna(0).astype("int64").astype(str)

# Kiểm tra lại kết quả
print(df[["children", "country", "agent"]].isnull().sum())

children    0
country     0
agent       0
dtype: int64


In [4]:

#? Danh sách các cột dùng làm "key" để xác định trùng lặp
key_columns = [
    'hotel',
    'lead_time',
    'arrival_date_year',
    'arrival_date_month',
    'arrival_date_day_of_month',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
    'adults',
    'children',
    'babies',
    'country',
    'adr'
]

#* Kiểm tra trùng lặp dựa trên nhóm cột trên
subset_duplicates = df.duplicated(subset=key_columns).sum()
print(f"Số dòng trùng lặp theo nhóm cột chỉ định: {subset_duplicates:,.0f}")

# Lọc ra các dòng đó
df[df.duplicated(subset=key_columns, keep=False)].sort_values(by=key_columns)

Số dòng trùng lặp theo nhóm cột chỉ định: 36,579


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40596,City Hotel,0,0,2015,August,32,3,1,0,2,...,0,No Deposit,0,0,Transient,0.0,0,0,Check-Out,4/8/2015 0:00
40598,City Hotel,0,0,2015,August,32,3,1,0,2,...,0,No Deposit,0,0,Transient,0.0,0,0,Check-Out,4/8/2015 0:00
40643,City Hotel,1,0,2015,August,32,4,0,1,2,...,0,No Deposit,0,0,Transient,0.0,0,0,Canceled,4/8/2015 0:00
40644,City Hotel,1,0,2015,August,32,4,0,1,2,...,0,No Deposit,0,0,Transient,0.0,0,0,Canceled,4/8/2015 0:00
40742,City Hotel,0,0,2015,August,32,6,0,1,2,...,1,No Deposit,0,0,Transient,81.0,0,0,Check-Out,7/8/2015 0:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8399,Resort Hotel,0,542,2016,September,40,26,2,5,2,...,0,No Deposit,253,0,Transient-Party,87.0,0,2,Check-Out,3/10/2016 0:00
8422,Resort Hotel,0,542,2016,September,40,26,2,5,2,...,0,No Deposit,253,0,Transient-Party,87.0,0,3,Check-Out,3/10/2016 0:00
8400,Resort Hotel,0,542,2016,September,40,26,2,5,2,...,0,No Deposit,253,0,Transient-Party,89.0,0,3,Check-Out,3/10/2016 0:00
8421,Resort Hotel,0,542,2016,September,40,26,2,5,2,...,0,No Deposit,253,0,Transient-Party,89.0,0,2,Check-Out,3/10/2016 0:00


In [5]:

#? Tạo ra một DataFrame mới đã sạch dữ liệu trùng
df_cleaned = df.drop_duplicates(keep="first")

#* Xóa trùng lặp theo tập hợp cột trên:
df_cleaned = df.drop_duplicates(subset=key_columns, keep='first')

print(f"Số dòng ban đầu: {len(df):,}")
print(f"Số dòng sau khi xóa trùng: {len(df_cleaned):,}")

Số dòng ban đầu: 119,390
Số dòng sau khi xóa trùng: 82,811


In [6]:

#? Xuất CSV đã clean với hậu tố _v2 vào thư mục data
output_path = csv_path.parent / f"{csv_path.stem}_v2.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df_cleaned.to_csv(output_path, index=False)

print(f"Đã xuất file: {output_path}")
print(f"Số dòng: {len(df_cleaned):,}, Số cột: {df_cleaned.shape[1]}")

Đã xuất file: C:\Users\ADMIN\OneDrive\Tài liệu\oneDrive\Desktop\DA\Project\Hotel Booking Demand\Python\data\hotel_bookings_v2.csv
Số dòng: 82,811, Số cột: 31


In [7]:
import pandas as pd

# Bước 1: Tạo cột phụ đánh dấu booking thành công (is_canceled == 0 thì mang giá trị 1)
df["is_successful"] = 1 - df["is_canceled"]

# Bước 2: Gom nhóm theo Khách sạn và Thời gian (Năm, Tháng)
time_groups = ["hotel", "arrival_date_year", "arrival_date_month"]
grouped = df.groupby(time_groups)

#* 1. Tính Occupancy Rate = (Tổng số booking is_canceled=0) / (Tổng số booking)
occupancy_rate = grouped["is_successful"].mean()

#* 2. Tính ADR trung bình (*LƯU Ý: Chỉ tính ADR trên các booking thực sự tới ở)
valid_adr = df[df["is_canceled"] == 0].groupby(time_groups)["adr"].mean()

# Bước 3: Tổng hợp vào bảng kết quả
revpar_report = pd.DataFrame(
    {
        "Total_Bookings": grouped.size(),
        "Occupancy_Rate": occupancy_rate,
        "ADR": valid_adr,
    }
).reset_index()

#* Bước 4: Áp dụng công thức RevPAR = ADR * Occupancy Rate
revpar_report["RevPAR"] = revpar_report["ADR"] * revpar_report["Occupancy_Rate"]

# Hiển thị thử 10 dòng đầu tiên
revpar_report.head(10)

,hotel,arrival_date_year,arrival_date_month,Total_Bookings,Occupancy_Rate,ADR,RevPAR
0,City Hotel,2015,August,2480,0.503226,81.740096,41.133726
1,City Hotel,2015,December,1654,0.596131,77.402860,46.142213
2,City Hotel,2015,July,1398,0.328326,73.591481,24.162010
3,City Hotel,2015,November,1235,0.756275,71.385300,53.986939
4,City Hotel,2015,October,3386,0.609864,92.148281,56.197933
5,City Hotel,2015,September,3529,0.562766,103.536143,58.266585
6,City Hotel,2016,April,3561,0.567818,99.399644,56.440910
7,City Hotel,2016,August,3378,0.630847,121.867733,76.879852
8,City Hotel,2016,December,2478,0.567393,95.187881,54.008943
9,City Hotel,2016,February,2371,0.607760,82.081020,49.885597


In [8]:
import pandas as pd

#? Gom nhóm theo Loại khách sạn và Thời gian
time_groups = ["hotel", "arrival_date_year", "arrival_date_month"]

#* Tính thống kê
cancellation_report = (
    df.groupby(time_groups)
    .agg(
        Total_Bookings=("is_canceled", "count"),
        Canceled_Bookings=("is_canceled", "sum"),
        Cancellation_Rate=("is_canceled", lambda x: x.mean() * 100),
    )
    .reset_index()
)

# Làm tròn tỷ lệ hủy lấy 2 chữ số thập phân
cancellation_report["Cancellation_Rate"] = cancellation_report[
    "Cancellation_Rate"
].round(2)

# Xem thử kết quả 10 dòng đầu
cancellation_report.head(10)

,hotel,arrival_date_year,arrival_date_month,Total_Bookings,Canceled_Bookings,Cancellation_Rate
0,City Hotel,2015,August,2480,1232,49.68
1,City Hotel,2015,December,1654,668,40.39
2,City Hotel,2015,July,1398,939,67.17
3,City Hotel,2015,November,1235,301,24.37
4,City Hotel,2015,October,3386,1321,39.01
5,City Hotel,2015,September,3529,1543,43.72
6,City Hotel,2016,April,3561,1539,43.22
7,City Hotel,2016,August,3378,1247,36.92
8,City Hotel,2016,December,2478,1072,43.26
9,City Hotel,2016,February,2371,930,39.22


In [9]:
# Tính 2 cột mới theo nhóm (hotel + năm + tháng) rồi gắn vào từng booking
df_v3 = df_cleaned.copy()

# Tránh lỗi khi chạy lại cell nhiều lần
for col in ["Occupancy_Rate", "RevPAR", "Group_ADR", "is_successful"]:
    if col in df_v3.columns:
        df_v3 = df_v3.drop(columns=[col])

time_groups = ["hotel", "arrival_date_year", "arrival_date_month"]

df_v3["is_successful"] = 1 - df_v3["is_canceled"]
grouped = df_v3.groupby(time_groups, as_index=False)

# Occupancy Rate: tỷ lệ booking không bị hủy trong từng nhóm
occupancy_rate = grouped["is_successful"].mean().rename(columns={"is_successful": "Occupancy_Rate"})

# ADR trung bình chỉ trên booking thực sự đến ở (is_canceled = 0)
valid_adr = (
    df_v3[df_v3["is_canceled"] == 0]
    .groupby(time_groups, as_index=False)["adr"]
    .mean()
    .rename(columns={"adr": "Group_ADR"})
)

metrics = occupancy_rate.merge(valid_adr, on=time_groups, how="inner")
metrics["RevPAR"] = metrics["Group_ADR"] * metrics["Occupancy_Rate"]

# Gắn chỉ số nhóm vào từng dòng booking
df_v3 = df_v3.merge(
    metrics[time_groups + ["Occupancy_Rate", "RevPAR"]],
    on=time_groups,
    how="left",
).drop(columns=["is_successful"])

# Kiểm tra: mỗi nhóm hotel-tháng phải có 1 giá trị Occupancy/RevPAR riêng
n_groups = metrics.shape[0]
n_occ = df_v3["Occupancy_Rate"].nunique()
n_revpar = df_v3["RevPAR"].nunique()

print(f"Số nhóm hotel-tháng: {n_groups}")
print(f"Số giá trị Occupancy_Rate khác nhau: {n_occ}")
print(f"Số giá trị RevPAR khác nhau: {n_revpar}")

if n_occ <= 1 or n_revpar <= 1:
    raise ValueError("Cột mới chỉ có 1 giá trị — kiểm tra lại bước groupby/merge.")

print("\nMẫu theo từng nhóm (mỗi nhóm 1 giá trị, các booking cùng nhóm sẽ giống nhau):")
display(
    df_v3.groupby(time_groups, as_index=False)
    .agg(
        bookings=("is_canceled", "count"),
        Occupancy_Rate=("Occupancy_Rate", "first"),
        RevPAR=("RevPAR", "first"),
    )
    .head(8)
)

# Xuất CSV với hậu tố _v3
output_path_v3 = csv_path.parent / f"{csv_path.stem}_v3.csv"
output_path_v3.parent.mkdir(parents=True, exist_ok=True)

df_v3.to_csv(output_path_v3, index=False)

print(f"\nĐã xuất file: {output_path_v3}")
print(f"Số dòng: {len(df_v3):,}, Số cột: {df_v3.shape[1]}")

Số nhóm hotel-tháng: 52
Số giá trị Occupancy_Rate khác nhau: 52
Số giá trị RevPAR khác nhau: 52

Mẫu theo từng nhóm (mỗi nhóm 1 giá trị, các booking cùng nhóm sẽ giống nhau):


,hotel,arrival_date_year,arrival_date_month,bookings,Occupancy_Rate,RevPAR
0,City Hotel,2015,August,994,0.787726,66.885292
1,City Hotel,2015,December,957,0.782654,63.898568
2,City Hotel,2015,July,331,0.383686,22.739909
3,City Hotel,2015,November,714,0.820728,59.917857
4,City Hotel,2015,October,1373,0.804079,78.170787
5,City Hotel,2015,September,1440,0.814583,88.884639
6,City Hotel,2016,April,2292,0.686300,69.913181
7,City Hotel,2016,August,2740,0.675912,84.164661



Đã xuất file: C:\Users\ADMIN\OneDrive\Tài liệu\oneDrive\Desktop\DA\Project\Hotel Booking Demand\Python\data\hotel_bookings_v3.csv
Số dòng: 82,811, Số cột: 33


In [10]:
from pathlib import Path
import pandas as pd

# 1. Tạo bản sao từ dữ liệu đã làm sạch
df_v4 = df_cleaned.copy()

# Tránh lỗi khi chạy lại cell nhiều lần
for col in [
    "Occupancy_Rate",
    "RevPAR",
    "Group_ADR",
    "is_successful",
    "total_nights",
    "revenue",
]:
    if col in df_v4.columns:
        df_v4 = df_v4.drop(columns=[col])

# =====================================================================
# PHẦN A: Tính toán 2 cột Row-level (total_nights & revenue)
# =====================================================================
df_v4["total_nights"] = (
    df_v4["stays_in_weekend_nights"] + df_v4["stays_in_week_nights"]
)

# Doanh thu thực tế (Nếu hủy phòng is_canceled == 1 thì doanh thu = 0)
df_v4["revenue"] = df_v4["adr"] * df_v4["total_nights"] * (1 - df_v4["is_canceled"])


# =====================================================================
# PHẦN B: Tính toán 2 cột Group KPI (Occupancy_Rate & RevPAR)
# =====================================================================
time_groups = ["hotel", "arrival_date_year", "arrival_date_month"]

df_v4["is_successful"] = 1 - df_v4["is_canceled"]
grouped = df_v4.groupby(time_groups, as_index=False)

# Occupancy Rate: tỷ lệ booking không bị hủy trong từng nhóm
occupancy_rate = grouped["is_successful"].mean().rename(
    columns={"is_successful": "Occupancy_Rate"}
)

# ADR trung bình chỉ trên booking thực sự đến ở (is_canceled == 0)
valid_adr = (
    df_v4[df_v4["is_canceled"] == 0]
    .groupby(time_groups, as_index=False)["adr"]
    .mean()
    .rename(columns={"adr": "Group_ADR"})
)

metrics = occupancy_rate.merge(valid_adr, on=time_groups, how="inner")
metrics["RevPAR"] = metrics["Group_ADR"] * metrics["Occupancy_Rate"]

# Gắn chỉ số nhóm vào từng dòng booking
df_v4 = df_v4.merge(
    metrics[time_groups + ["Occupancy_Rate", "RevPAR"]],
    on=time_groups,
    how="left",
).drop(columns=["is_successful"])


# =====================================================================
# PHẦN C: Kiểm tra tính toàn vẹn dữ liệu
# =====================================================================
n_groups = metrics.shape[0]
n_occ = df_v4["Occupancy_Rate"].nunique()
n_revpar = df_v4["RevPAR"].nunique()

print(f"Số nhóm hotel-tháng: {n_groups}")
print(f"Số giá trị Occupancy_Rate khác nhau: {n_occ}")
print(f"Số giá trị RevPAR khác nhau: {n_revpar}")

if n_occ <= 1 or n_revpar <= 1:
    raise ValueError("Cột mới chỉ có 1 giá trị — kiểm tra lại bước groupby/merge.")

print("\nMẫu tổng hợp theo từng nhóm (kiểm tra cả 4 cột mới bổ sung):")
display(
    df_v4.groupby(time_groups, as_index=False)
    .agg(
        bookings=("is_canceled", "count"),
        avg_nights=("total_nights", "mean"),
        total_group_rev=("revenue", "sum"),
        Occupancy_Rate=("Occupancy_Rate", "first"),
        RevPAR=("RevPAR", "first"),
    )
    .head(8)
)


# =====================================================================
# PHẦN D: Xuất CSV với hậu tố _v4 vào thư mục data (cùng nơi file gốc)
# =====================================================================
output_path_v4 = csv_path.parent / f"{csv_path.stem}_v4.csv"
output_path_v4.parent.mkdir(parents=True, exist_ok=True)

df_v4.to_csv(output_path_v4, index=False)

print(f"\nĐã xuất file: {output_path_v4.resolve()}")
print(f"Số dòng: {len(df_v4):,}, Số cột: {df_v4.shape[1]}")

Số nhóm hotel-tháng: 52
Số giá trị Occupancy_Rate khác nhau: 52
Số giá trị RevPAR khác nhau: 52

Mẫu tổng hợp theo từng nhóm (kiểm tra cả 4 cột mới bổ sung):


,hotel,arrival_date_year,arrival_date_month,bookings,avg_nights,total_group_rev,Occupancy_Rate,RevPAR
0,City Hotel,2015,August,994,3.067404,196378.27,0.787726,66.885292
1,City Hotel,2015,December,957,3.074190,180064.54,0.782654,63.898568
2,City Hotel,2015,July,331,3.631420,24014.51,0.383686,22.739909
3,City Hotel,2015,November,714,3.057423,127706.01,0.820728,59.917857
4,City Hotel,2015,October,1373,2.893664,316163.36,0.804079,78.170787
5,City Hotel,2015,September,1440,2.913194,384546.43,0.814583,88.884639
6,City Hotel,2016,April,2292,3.072862,479458.23,0.686300,69.913181
7,City Hotel,2016,August,2740,3.433942,725485.64,0.675912,84.164661



Đã xuất file: C:\Users\ADMIN\OneDrive\Tài liệu\oneDrive\Desktop\DA\Project\Hotel Booking Demand\Python\data\hotel_bookings_v4.csv
Số dòng: 82,811, Số cột: 35


In [11]:
# Kiểm tra tổng quan lý do revenue = 0
zero_rev = df_v4[df_v4["revenue"] == 0]

print(f"Tổng số dòng có Doanh thu = 0: {len(zero_rev):,}")
print("-" * 40)
print(
    "1. Số dòng bằng 0 do KHÁCH HỦY PHÒNG:",
    len(zero_rev[zero_rev["is_canceled"] == 1]),
)
print(
    "2. Số dòng bằng 0 do PHÒNG MIỄN PHÍ (ADR=0):",
    len(zero_rev[(zero_rev["is_canceled"] == 0) & (zero_rev["adr"] == 0)]),
)
print(
    "3. Số dòng bằng 0 do KHÔNG Ở ĐÊM NÀO (Nights=0):",
    len(zero_rev[(zero_rev["is_canceled"] == 0) & (zero_rev["total_nights"] == 0)]),
)

Tổng số dòng có Doanh thu = 0: 24,744
----------------------------------------
1. Số dòng bằng 0 do KHÁCH HỦY PHÒNG: 23284
2. Số dòng bằng 0 do PHÒNG MIỄN PHÍ (ADR=0): 1460
3. Số dòng bằng 0 do KHÔNG Ở ĐÊM NÀO (Nights=0): 584


In [12]:
# Tập tổng: Các dòng có revenue = 0
zero_rev = df_v4[df_v4["revenue"] == 0]

# Nhóm 1: Khách hủy phòng
g1 = zero_rev[zero_rev["is_canceled"] == 1]

# Nhóm 2: Khách tới ở VÀ có số đêm > 0 VÀ phòng miễn phí (ADR = 0)
g2 = zero_rev[
    (zero_rev["is_canceled"] == 0)
    & (zero_rev["total_nights"] > 0)
    & (zero_rev["adr"] == 0)
]

# Nhóm 3: Khách tới ở nhưng số đêm lưu trú = 0
g3 = zero_rev[(zero_rev["is_canceled"] == 0) & (zero_rev["total_nights"] == 0)]

print(f"Tổng thực tế dòng revenue = 0 : {len(zero_rev):,}")
print("=" * 45)
print(f"• Nhóm 1 (Hủy phòng)                : {len(g1):,}")
print(f"• Nhóm 2 (Ở thật nhưng miễn phí)     : {len(g2):,}")
print(f"• Nhóm 3 (Lỗi / Day-use 0 đêm)       : {len(g3):,}")
print("-" * 45)
print(f"☞ Tổng cộng 3 nhóm tách biệt        : {len(g1) + len(g2) + len(g3):,}")

Tổng thực tế dòng revenue = 0 : 24,744
• Nhóm 1 (Hủy phòng)                : 23,284
• Nhóm 2 (Ở thật nhưng miễn phí)     : 876
• Nhóm 3 (Lỗi / Day-use 0 đêm)       : 584
---------------------------------------------
☞ Tổng cộng 3 nhóm tách biệt        : 24,744


# Giao tập hợp (Phồng đếm do bị trùng lặp)Nguyên nhân không phải do mất dữ liệu, mà là do một đơn đặt phòng có thể đồng thời thỏa mãn 2 điều kiện cùng lúc.
Trong code kiểm tra ở bước trước, chúng ta tách riêng:
Nhóm 2: Khách tới ở (is_canceled = 0) nhưng Giá phòng bằng 0 (adr = 0).
Nhóm 3: Khách tới ở (is_canceled = 0) nhưng Số đêm bằng 0 (total_nights = 0).
Trong thực tế hệ thống khách sạn, toàn bộ 540 đơn không ở đêm nào (Nights = 0) của Nhóm 3 cũng đồng thời bị hệ thống gõ giá phòng bằng 0 (ADR = 0).
Khi bạn đếm độc lập từng nhóm, 540 đơn này đã được tính một lần ở Nhóm 2 rồi lại được tính thêm một lần nữa ở Nhóm 3 -> Dẫn đến tổng bị phồng lên 540 đơn vị!